In [8]:
import numpy as np
from datetime import datetime
import plotly.graph_objects as go

# --------------------------------
# Data: first consensus sport route of each grade
# --------------------------------
grade_labels = ["8a+", "8b", "8b+", "8c", "8c+", "9a", "9a+", "9b", "9b+", "9c"]
x_actual = np.arange(len(grade_labels) - 1)  # 0..8
x_all = np.arange(len(grade_labels))         # 0..9

dates = [
    datetime(1983, 1, 1),   # 8a+ The Face
    datetime(1984, 1, 1),   # 8b Kanal im Rücken
    datetime(1985, 1, 1),   # 8b+ Punks in the Gym
    datetime(1987, 1, 1),   # 8c Wallstreet
    datetime(1990, 6, 1),   # 8c+ Hubble
    datetime(1991, 1, 1),   # 9a Action Directe
    datetime(2001, 7, 1),   # 9a+ Realization
    datetime(2008, 9, 11),  # 9b Jumbo Love
    datetime(2012, 10, 4),  # 9b+ Change
]

labels = [
    "1983<br>8a+",
    "1984<br>8b",
    "1985<br>8b+",
    "1987<br>8c",
    "Jun 1990<br>8c+",
    "1991<br>9a",
    "Jul 2001<br>9a+",
    "Sep 2008<br>9b",
    "Oct 2012<br>9b+",
]

def decimal_year(dt):
    year_start = datetime(dt.year, 1, 1)
    next_year = datetime(dt.year + 1, 1, 1)
    return dt.year + (dt - year_start).total_seconds() / (next_year - year_start).total_seconds()

years = np.array([decimal_year(d) for d in dates])

# --------------------------------
# Model 1: direct linear fit
# year = a + b * step
# --------------------------------
coef_direct = np.polyfit(x_actual, years, 1)
poly_direct = np.poly1d(coef_direct)

# --------------------------------
# Model 2: linear growth in gaps
# --------------------------------
gaps = np.diff(years)
steps = np.arange(1, len(gaps) + 1, dtype=float)

coef_gap_linear = np.polyfit(steps, gaps, 1)
poly_gap_linear = np.poly1d(coef_gap_linear)

def predict_gap_linear(target_index):
    predicted = [years[0]]
    for idx in range(1, target_index + 1):
        gap = poly_gap_linear(idx)
        predicted.append(predicted[-1] + gap)
    return np.array(predicted)

# --------------------------------
# Model 3: exponential growth in gaps
# --------------------------------
log_gaps = np.log(gaps)
coef_gap_exp = np.polyfit(steps, log_gaps, 1)

def predict_gap_exponential(target_index):
    predicted = [years[0]]
    for idx in range(1, target_index + 1):
        gap = np.exp(coef_gap_exp[1] + coef_gap_exp[0] * idx)
        predicted.append(predicted[-1] + gap)
    return np.array(predicted)

# --------------------------------
# Build curves through 9c
# --------------------------------
y_direct = poly_direct(x_all)
y_gap_linear = predict_gap_linear(x_all[-1])
y_gap_exp = predict_gap_exponential(x_all[-1])

pred_direct = y_direct[-1]
pred_gap_linear = y_gap_linear[-1]
pred_gap_exp = y_gap_exp[-1]

# --------------------------------
# Plot
# --------------------------------
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_actual,
    y=years,
    mode="markers+text",
    name="Actual first consensus redpoint",
    text=labels,
    textposition=[
        "bottom left",   # 8a+
        "top left",      # 8b
        "bottom center", # 8b+
        "top center",    # 8c
        "bottom right",  # 8c+
        "top left",      # 9a
        "bottom center", # 9a+
        "top center",    # 9b
        "bottom right",  # 9b+
    ],
    marker=dict(size=12),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_direct,
    mode="lines+markers",
    name=f"Direct linear fit (9c ≈ {pred_direct:.1f})",
    line=dict(dash="dash"),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_linear,
    mode="lines+markers",
    name=f"Linear-gap model (9c ≈ {pred_gap_linear:.1f})",
    line=dict(dash="dot"),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_exp,
    mode="lines+markers",
    name=f"Exponential-gap model (9c ≈ {pred_gap_exp:.1f})",
    line=dict(dash="longdash"),
))

fig.add_annotation(x=9, y=pred_direct, text=f"{pred_direct:.1f}", showarrow=True, ax=40, ay=-20)
fig.add_annotation(x=9, y=pred_gap_linear, text=f"{pred_gap_linear:.1f}", showarrow=True, ax=55, ay=-65)
fig.add_annotation(x=9, y=pred_gap_exp, text=f"{pred_gap_exp:.1f}", showarrow=True, ax=55, ay=-95)

fig.update_layout(
    title="When Was 9c Expected to Arrive?",
    template="plotly_dark",
    xaxis=dict(
        title="Sport Grade",
        tickmode="array",
        tickvals=x_all,
        ticktext=grade_labels,
        range=[-0.4, 9.9],
    ),
    yaxis=dict(
        title="Year",
        range=[1981, max(pred_direct, pred_gap_linear, pred_gap_exp) + 2],
    ),
    legend=dict(
        title="Projection model",
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
    ),
    margin=dict(r=260),
    width=1150,
    height=650,
)

fig.show()

print(f"Direct linear fit 9c: {pred_direct:.2f}")
print(f"Linear-gap model 9c: {pred_gap_linear:.2f}")
print(f"Exponential-gap model 9c: {pred_gap_exp:.2f}")

Direct linear fit 9c: 2012.88
Linear-gap model 9c: 2020.50
Exponential-gap model 9c: 2014.78


In [9]:
import numpy as np
from datetime import datetime
import plotly.graph_objects as go

# --------------------------------
# Data: first consensus / frontier sport route of each grade
# treating Silence as the first 9c
# --------------------------------
grade_labels = ["8a+", "8b", "8b+", "8c", "8c+", "9a", "9a+", "9b", "9b+", "9c", "9c+"]
x_actual = np.arange(len(grade_labels) - 1)   # 0..9
x_all = np.arange(len(grade_labels))          # 0..10

dates = [
    datetime(1983, 1, 1),   # 8a+ The Face
    datetime(1984, 1, 1),   # 8b Kanal im Rücken
    datetime(1985, 1, 1),   # 8b+ Punks in the Gym
    datetime(1987, 1, 1),   # 8c Wallstreet
    datetime(1990, 6, 1),   # 8c+ Hubble
    datetime(1991, 1, 1),   # 9a Action Directe
    datetime(2001, 7, 1),   # 9a+ Realization
    datetime(2008, 9, 11),  # 9b Jumbo Love
    datetime(2012, 10, 4),  # 9b+ Change
    datetime(2017, 9, 3),   # 9c Silence
]

labels = [
    "1983<br>8a+",
    "1984<br>8b",
    "1985<br>8b+",
    "1987<br>8c",
    "Jun 1990<br>8c+",
    "1991<br>9a",
    "Jul 2001<br>9a+",
    "Sep 2008<br>9b",
    "Oct 2012<br>9b+",
    "Sep 2017<br>9c",
]

def decimal_year(dt):
    year_start = datetime(dt.year, 1, 1)
    next_year = datetime(dt.year + 1, 1, 1)
    return dt.year + (dt - year_start).total_seconds() / (next_year - year_start).total_seconds()

years = np.array([decimal_year(d) for d in dates])

# --------------------------------
# Model 1: direct linear fit
# --------------------------------
coef_direct = np.polyfit(x_actual, years, 1)
poly_direct = np.poly1d(coef_direct)

# --------------------------------
# Model 2: linear growth in gaps
# --------------------------------
gaps = np.diff(years)
steps = np.arange(1, len(gaps) + 1, dtype=float)

coef_gap_linear = np.polyfit(steps, gaps, 1)
poly_gap_linear = np.poly1d(coef_gap_linear)

def predict_gap_linear(target_index):
    predicted = [years[0]]
    for idx in range(1, target_index + 1):
        gap = poly_gap_linear(idx)
        predicted.append(predicted[-1] + gap)
    return np.array(predicted)

# --------------------------------
# Model 3: exponential growth in gaps
# --------------------------------
log_gaps = np.log(gaps)
coef_gap_exp = np.polyfit(steps, log_gaps, 1)

def predict_gap_exponential(target_index):
    predicted = [years[0]]
    for idx in range(1, target_index + 1):
        gap = np.exp(coef_gap_exp[1] + coef_gap_exp[0] * idx)
        predicted.append(predicted[-1] + gap)
    return np.array(predicted)

# --------------------------------
# Build curves through 9c+
# --------------------------------
y_direct = poly_direct(x_all)
y_gap_linear = predict_gap_linear(x_all[-1])
y_gap_exp = predict_gap_exponential(x_all[-1])

pred_direct = y_direct[-1]
pred_gap_linear = y_gap_linear[-1]
pred_gap_exp = y_gap_exp[-1]

# --------------------------------
# Plot
# --------------------------------
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_actual,
    y=years,
    mode="markers+text",
    name="Actual first ascent",
    text=labels,
    textposition=[
        "bottom left",   # 8a+
        "top left",      # 8b
        "bottom center", # 8b+
        "top center",    # 8c
        "bottom right",  # 8c+
        "top left",      # 9a
        "bottom center", # 9a+
        "top center",    # 9b
        "bottom right",  # 9b+
        "top center",    # 9c
    ],
    marker=dict(size=12),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_direct,
    mode="lines+markers",
    name=f"Direct linear fit (9c+ ≈ {pred_direct:.1f})",
    line=dict(dash="dash"),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_linear,
    mode="lines+markers",
    name=f"Linear-gap model (9c+ ≈ {pred_gap_linear:.1f})",
    line=dict(dash="dot"),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_exp,
    mode="lines+markers",
    name=f"Exponential-gap model (9c+ ≈ {pred_gap_exp:.1f})",
    line=dict(dash="longdash"),
))

fig.add_annotation(
    x=10, y=pred_direct,
    text=f"{pred_direct:.1f}",
    showarrow=True,
    ax=45, ay=-20,
    bgcolor="rgba(0,0,0,0.7)"
)

fig.add_annotation(
    x=10, y=pred_gap_linear,
    text=f"{pred_gap_linear:.1f}",
    showarrow=True,
    ax=65, ay=-70,
    bgcolor="rgba(0,0,0,0.7)"
)

fig.add_annotation(
    x=10, y=pred_gap_exp,
    text=f"{pred_gap_exp:.1f}",
    showarrow=True,
    ax=65, ay=-105,
    bgcolor="rgba(0,0,0,0.7)"
)

fig.update_layout(
    title="When Will the First 9c+ Sport Route Be Redpointed?",
    template="plotly_dark",
    xaxis=dict(
        title="Sport Grade",
        tickmode="array",
        tickvals=x_all,
        ticktext=grade_labels,
        range=[-0.4, 10.7],
    ),
    yaxis=dict(
        title="Year",
        range=[1981, max(pred_direct, pred_gap_linear, pred_gap_exp) + 2],
    ),
    legend=dict(
        title="Projection model",
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
    ),
    margin=dict(r=260),
    width=1150,
    height=650,
)

fig.show(renderer="browser")

print(f"Direct linear fit 9c+: {pred_direct:.2f}")
print(f"Linear-gap model 9c+: {pred_gap_linear:.2f}")
print(f"Exponential-gap model 9c+: {pred_gap_exp:.2f}")

Direct linear fit 9c+: 2018.63
Linear-gap model 9c+: 2025.05
Exponential-gap model 9c+: 2020.02


In [10]:
import numpy as np
from datetime import datetime
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

# --------------------------------
# Frontier progression only
# --------------------------------
grade_labels = ["9a+", "9b", "9b+", "9c", "9c+"]
x_actual = np.arange(len(grade_labels) - 1)  # 0..3
x_all = np.arange(len(grade_labels))         # 0..4

dates = [
    datetime(2001, 7, 1),   # 9a+ Realization
    datetime(2008, 9, 11),  # 9b Jumbo Love
    datetime(2012, 10, 4),  # 9b+ Change
    datetime(2017, 9, 3),   # 9c Silence
]

labels = [
    "Jul 2001<br>9a+",
    "Sep 2008<br>9b",
    "Oct 2012<br>9b+",
    "Sep 2017<br>9c",
]

def decimal_year(dt):
    year_start = datetime(dt.year,1,1)
    next_year = datetime(dt.year+1,1,1)
    return dt.year + (dt-year_start).total_seconds()/(next_year-year_start).total_seconds()

years = np.array([decimal_year(d) for d in dates])

# --------------------------------
# Model 1: direct linear fit
# --------------------------------
coef_direct = np.polyfit(x_actual, years, 1)
poly_direct = np.poly1d(coef_direct)

# --------------------------------
# Model 2: linear gap growth
# --------------------------------
gaps = np.diff(years)
steps = np.arange(1, len(gaps)+1, dtype=float)

coef_gap_linear = np.polyfit(steps, gaps, 1)
poly_gap_linear = np.poly1d(coef_gap_linear)

def predict_gap_linear(target_index):
    predicted=[years[0]]
    for i in range(1, target_index+1):
        gap=poly_gap_linear(i)
        predicted.append(predicted[-1]+gap)
    return np.array(predicted)

# --------------------------------
# Model 3: exponential gap growth
# --------------------------------
log_gaps = np.log(gaps)
coef_gap_exp = np.polyfit(steps, log_gaps, 1)

def predict_gap_exponential(target_index):
    predicted=[years[0]]
    for i in range(1, target_index+1):
        gap=np.exp(coef_gap_exp[1] + coef_gap_exp[0]*i)
        predicted.append(predicted[-1]+gap)
    return np.array(predicted)

# --------------------------------
# Build predictions
# --------------------------------
y_direct = poly_direct(x_all)
y_gap_linear = predict_gap_linear(x_all[-1])
y_gap_exp = predict_gap_exponential(x_all[-1])

pred_direct = y_direct[-1]
pred_gap_linear = y_gap_linear[-1]
pred_gap_exp = y_gap_exp[-1]

# --------------------------------
# Plot
# --------------------------------
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_actual,
    y=years,
    mode="markers+text",
    name="Actual first ascent",
    text=labels,
    textposition=[
        "bottom left",
        "top center",
        "bottom right",
        "top center"
    ],
    marker=dict(size=12),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_direct,
    mode="lines+markers",
    name=f"Direct linear fit (9c+ ≈ {pred_direct:.1f})",
    line=dict(dash="dash"),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_linear,
    mode="lines+markers",
    name=f"Linear-gap model (9c+ ≈ {pred_gap_linear:.1f})",
    line=dict(dash="dot"),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_exp,
    mode="lines+markers",
    name=f"Exponential-gap model (9c+ ≈ {pred_gap_exp:.1f})",
    line=dict(dash="longdash"),
))

fig.add_annotation(
    x=4, y=pred_direct,
    text=f"{pred_direct:.1f}",
    showarrow=True,
    ax=40, ay=-20,
    bgcolor="rgba(0,0,0,0.7)"
)

fig.add_annotation(
    x=4, y=pred_gap_linear,
    text=f"{pred_gap_linear:.1f}",
    showarrow=True,
    ax=60, ay=-70,
    bgcolor="rgba(0,0,0,0.7)"
)

fig.add_annotation(
    x=4, y=pred_gap_exp,
    text=f"{pred_gap_exp:.1f}",
    showarrow=True,
    ax=60, ay=-100,
    bgcolor="rgba(0,0,0,0.7)"
)

fig.update_layout(
    title="When Will the First 9c+ Sport Route Be Climbed?",
    template="plotly_dark",
    xaxis=dict(
        title="Sport Grade",
        tickmode="array",
        tickvals=x_all,
        ticktext=grade_labels,
        range=[-0.3,4.7],
    ),
    yaxis=dict(
        title="Year",
        range=[1998, max(pred_direct,pred_gap_linear,pred_gap_exp)+2],
    ),
    legend=dict(
        title="Projection model",
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top"
    ),
    margin=dict(r=260),
    width=1100,
    height=650
)

fig.show()

In [16]:
import numpy as np
from datetime import datetime
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

# --------------------------------
# Sport flash/onsight frontier
# treating flash and onsight as the same:
# use whichever came first for each grade
# --------------------------------
grade_labels = ["7b+", "7c", "7c+", "8a", "8a+", "8b", "8b+", "8c", "8c+", "9a", "9a+", "9b"]
x_all = np.arange(len(grade_labels))  # 0..11

# known frontier points at their true grade-step positions
# 8a+ is still unknown for men, so we skip x=4
x_actual = np.array([0, 1, 2, 3, 5, 6, 7, 8, 9, 10], dtype=float)

dates = [
    datetime(1982, 1, 1),    # 7b+ Super Crack / Captain crochet
    datetime(1982, 1, 1),    # 7c Equinox / La Polka des Ringards
    datetime(1984, 1, 1),    # 7c+ Pol Pot
    datetime(1987, 1, 1),    # 8a Samizdat
    datetime(1993, 1, 1),    # 8b Liaisons Dangereuses
    datetime(1995, 1, 1),    # 8b+ Massey Fergusson
    datetime(2004, 10, 6),   # 8c White Zombie
    datetime(2007, 12, 11),  # 8c+ Bizi Euskaraz
    datetime(2012, 10, 29),  # 9a Southern Smoke Direct (flash)
    datetime(2018, 2, 10),   # 9a+ Super Crackinette (flash)
]

labels = [
    "1982<br>7b+",
    "1982<br>7c",
    "1984<br>7c+",
    "1987<br>8a",
    "1993<br>8b",
    "1995<br>8b+",
    "Oct 2004<br>8c",
    "Dec 2007<br>8c+",
    "Oct 2012<br>9a",
    "Feb 2018<br>9a+",
]

def decimal_year(dt):
    year_start = datetime(dt.year, 1, 1)
    next_year = datetime(dt.year + 1, 1, 1)
    return dt.year + (dt - year_start).total_seconds() / (next_year - year_start).total_seconds()

years = np.array([decimal_year(d) for d in dates])

# --------------------------------
# Model 1: direct linear fit
# --------------------------------
coef_direct = np.polyfit(x_actual, years, 1)
poly_direct = np.poly1d(coef_direct)

# --------------------------------
# Model 2: linear growth in gaps
# uses per-step gap so missing 8a+ doesn't distort spacing
# --------------------------------
step_gaps = np.diff(years) / np.diff(x_actual)
steps = np.arange(1, len(step_gaps) + 1, dtype=float)

coef_gap_linear = np.polyfit(steps, step_gaps, 1)
poly_gap_linear = np.poly1d(coef_gap_linear)

def predict_gap_linear(target_x):
    predicted = [years[0]]
    current_x = int(x_actual[0])

    for next_x in range(current_x + 1, int(target_x) + 1):
        step_idx = next_x - current_x
        gap_per_step = poly_gap_linear(step_idx)
        predicted.append(predicted[-1] + gap_per_step)

    return np.array(predicted)

# --------------------------------
# Model 3: exponential growth in per-step gaps
# --------------------------------
log_gaps = np.log(step_gaps)
coef_gap_exp = np.polyfit(steps, log_gaps, 1)

def predict_gap_exponential(target_x):
    predicted = [years[0]]
    current_x = int(x_actual[0])

    for next_x in range(current_x + 1, int(target_x) + 1):
        step_idx = next_x - current_x
        gap_per_step = np.exp(coef_gap_exp[1] + coef_gap_exp[0] * step_idx)
        predicted.append(predicted[-1] + gap_per_step)

    return np.array(predicted)

# --------------------------------
# Build curves through 9b
# --------------------------------
y_direct = poly_direct(x_all)
y_gap_linear = predict_gap_linear(x_all[-1])
y_gap_exp = predict_gap_exponential(x_all[-1])

pred_direct = y_direct[-1]
pred_gap_linear = y_gap_linear[-1]
pred_gap_exp = y_gap_exp[-1]

# --------------------------------
# Plot
# --------------------------------
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_actual,
    y=years,
    mode="markers+text",
    name="Actual first flash/onsight",
    text=labels,
    textposition=[
        "bottom left",   # 7b+
        "top center",    # 7c
        "bottom center", # 7c+
        "top left",      # 8a
        "bottom center", # 8b
        "top left",      # 8b+
        "bottom right",  # 8c
        "top center",    # 8c+
        "bottom right",  # 9a
        "top center",    # 9a+
    ],
    marker=dict(size=12),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_direct,
    mode="lines+markers",
    name=f"Direct linear fit (9b ≈ {pred_direct:.1f})",
    line=dict(dash="dash"),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_linear,
    mode="lines+markers",
    name=f"Linear-gap model (9b ≈ {pred_gap_linear:.1f})",
    line=dict(dash="dot"),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_exp,
    mode="lines+markers",
    name=f"Exponential-gap model (9b ≈ {pred_gap_exp:.1f})",
    line=dict(dash="longdash"),
))

fig.add_annotation(
    x=11, y=pred_direct,
    text=f"{pred_direct:.1f}",
    showarrow=True,
    ax=45, ay=-20,
    bgcolor="rgba(0,0,0,0.7)"
)

fig.add_annotation(
    x=11, y=pred_gap_linear,
    text=f"{pred_gap_linear:.1f}",
    showarrow=True,
    ax=65, ay=-70,
    bgcolor="rgba(0,0,0,0.7)"
)

fig.add_annotation(
    x=11, y=pred_gap_exp,
    text=f"{pred_gap_exp:.1f}",
    showarrow=True,
    ax=65, ay=-105,
    bgcolor="rgba(0,0,0,0.7)"
)

fig.update_layout(
    title="When Will the First 9b Sport Route Be Flashed or Onsighted?",
    template="plotly_dark",
    xaxis=dict(
        title="Sport Grade",
        tickmode="array",
        tickvals=x_all,
        ticktext=grade_labels,
        range=[-0.3, 11.7],
    ),
    yaxis=dict(
        title="Year",
        range=[1980, max(pred_direct, pred_gap_linear, pred_gap_exp) + 2],
    ),
    legend=dict(
        title="Projection model",
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
    ),
    margin=dict(r=260),
    width=1150,
    height=650,
)

fig.show(renderer="browser")

print(f"Direct linear fit 9b flash/onsight: {pred_direct:.2f}")
print(f"Linear-gap model 9b flash/onsight: {pred_gap_linear:.2f}")
print(f"Exponential-gap model 9b flash/onsight: {pred_gap_exp:.2f}")

Direct linear fit 9b flash/onsight: 2018.85
Linear-gap model 9b flash/onsight: 2029.23
Exponential-gap model 9b flash/onsight: nan


C:\Users\joeka\AppData\Local\Temp\ipykernel_35892\1977646619.py:83: RuntimeWarning:

divide by zero encountered in log



In [13]:
import numpy as np
from datetime import datetime
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

# --------------------------------
# Free solo frontier
# --------------------------------
grade_labels = ["6c", "6c+", "7a", "7a+", "7b", "7c", "8a", "8a+", "8b", "8b+", "8c", "8c+"]
x_actual = np.arange(len(grade_labels) - 1)   # 0..10
x_all = np.arange(len(grade_labels))          # 0..11

dates = [
    datetime(1958, 7, 9),   # 6c   Western Front
    datetime(1961, 1, 1),   # 6c+  Wall of Horrors
    datetime(1978, 2, 1),   # 7a   Spider Line
    datetime(1961, 1, 1),   # 7a+  Thimble
    datetime(1981, 10, 1),  # 7b   Mickey's Beach Crack
    datetime(1986, 1, 1),   # 7c   Weed Killer
    datetime(1985, 1, 1),   # 8a   Revelations
    datetime(1987, 1, 1),   # 8a+  Rêve de gosse
    datetime(1993, 1, 1),   # 8b   Compilation
    datetime(2004, 1, 1),   # 8b+  Kommunist
    datetime(2021, 3, 1),   # 8c   Panem et Circenses
]

labels = [
    "Jul 1958<br>6c",
    "1961<br>6c+",
    "Feb 1978<br>7a",
    "1961<br>7a+",
    "Oct 1981<br>7b",
    "1986<br>7c",
    "1985<br>8a",
    "1987<br>8a+",
    "1993<br>8b",
    "2004<br>8b+",
    "Mar 2021<br>8c",
]

def decimal_year(dt):
    year_start = datetime(dt.year, 1, 1)
    next_year = datetime(dt.year + 1, 1, 1)
    return dt.year + (dt - year_start).total_seconds() / (next_year - year_start).total_seconds()

years = np.array([decimal_year(d) for d in dates])

# --------------------------------
# Model 1: direct linear fit
# --------------------------------
coef_direct = np.polyfit(x_actual, years, 1)
poly_direct = np.poly1d(coef_direct)

# --------------------------------
# Model 2: linear growth in gaps
# --------------------------------
gaps = np.diff(years)
steps = np.arange(1, len(gaps) + 1, dtype=float)

coef_gap_linear = np.polyfit(steps, gaps, 1)
poly_gap_linear = np.poly1d(coef_gap_linear)

def predict_gap_linear(target_index):
    predicted = [years[0]]
    for i in range(1, target_index + 1):
        gap = poly_gap_linear(i)
        predicted.append(predicted[-1] + gap)
    return np.array(predicted)

# --------------------------------
# Model 3: exponential growth in gaps
# --------------------------------
# shift if any non-positive gaps exist
min_gap = np.min(gaps)
gap_shift = 0.0
if min_gap <= 0:
    gap_shift = abs(min_gap) + 0.25

log_gaps = np.log(gaps + gap_shift)
coef_gap_exp = np.polyfit(steps, log_gaps, 1)

def predict_gap_exponential(target_index):
    predicted = [years[0]]
    for i in range(1, target_index + 1):
        gap = np.exp(coef_gap_exp[1] + coef_gap_exp[0] * i) - gap_shift
        predicted.append(predicted[-1] + gap)
    return np.array(predicted)

# --------------------------------
# Build curves through 8c+
# --------------------------------
y_direct = poly_direct(x_all)
y_gap_linear = predict_gap_linear(x_all[-1])
y_gap_exp = predict_gap_exponential(x_all[-1])

pred_direct = y_direct[-1]
pred_gap_linear = y_gap_linear[-1]
pred_gap_exp = y_gap_exp[-1]

# --------------------------------
# Plot
# --------------------------------
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_actual,
    y=years,
    mode="markers+text",
    name="Actual first free solo",
    text=labels,
    textposition=[
        "bottom left",   # 6c
        "top left",      # 6c+
        "bottom center", # 7a
        "top center",    # 7a+
        "bottom right",  # 7b
        "top center",    # 7c
        "bottom left",   # 8a
        "top left",      # 8a+
        "bottom center", # 8b
        "top center",    # 8b+
        "bottom right",  # 8c
    ],
    marker=dict(size=12),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_direct,
    mode="lines+markers",
    name=f"Direct linear fit (8c+ ≈ {pred_direct:.1f})",
    line=dict(dash="dash"),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_linear,
    mode="lines+markers",
    name=f"Linear-gap model (8c+ ≈ {pred_gap_linear:.1f})",
    line=dict(dash="dot"),
))

fig.add_trace(go.Scatter(
    x=x_all,
    y=y_gap_exp,
    mode="lines+markers",
    name=f"Exponential-gap model (8c+ ≈ {pred_gap_exp:.1f})",
    line=dict(dash="longdash"),
))

fig.add_annotation(
    x=11, y=pred_direct,
    text=f"{pred_direct:.1f}",
    showarrow=True,
    ax=45, ay=-20,
    bgcolor="rgba(0,0,0,0.7)"
)

fig.add_annotation(
    x=11, y=pred_gap_linear,
    text=f"{pred_gap_linear:.1f}",
    showarrow=True,
    ax=65, ay=-70,
    bgcolor="rgba(0,0,0,0.7)"
)

fig.add_annotation(
    x=11, y=pred_gap_exp,
    text=f"{pred_gap_exp:.1f}",
    showarrow=True,
    ax=65, ay=-105,
    bgcolor="rgba(0,0,0,0.7)"
)

fig.update_layout(
    title="When Will the First 8c+ Free Solo Happen?",
    template="plotly_dark",
    xaxis=dict(
        title="Route Grade",
        tickmode="array",
        tickvals=x_all,
        ticktext=grade_labels,
        range=[-0.3, 11.7],
    ),
    yaxis=dict(
        title="Year",
        range=[1955, max(pred_direct, pred_gap_linear, pred_gap_exp) + 2],
    ),
    legend=dict(
        title="Projection model",
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
    ),
    margin=dict(r=260),
    width=1150,
    height=650,
)

fig.show(renderer="browser")

print(f"Direct linear fit 8c+ free solo: {pred_direct:.2f}")
print(f"Linear-gap model 8c+ free solo: {pred_gap_linear:.2f}")
print(f"Exponential-gap model 8c+ free solo: {pred_gap_exp:.2f}")

Direct linear fit 8c+ free solo: 2015.24
Linear-gap model 8c+ free solo: 2032.21
Exponential-gap model 8c+ free solo: 1975.98
